# Three-Way Comparison: EBM vs NAM vs GAM

Statistical comparison of EBM, NAM, and GAM performance across 10 regression and 10 classification datasets.

This notebook:
1. Displays performance summary tables for all three models
2. Performs Friedman test to check if models differ significantly
3. If significant, performs Nemenyi post-hoc test to identify which model is best


In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp

# Project root
project_root = Path().resolve().parent.parent
results_dir = project_root / 'results' / 'evaluation'


In [2]:
# Dataset IDs
regression_datasets = [41021, 44956, 44957, 44958, 44959, 44966, 44972, 44980, 44987, 45402]
classification_datasets = [31, 37, 50, 1049, 1050, 1063, 1067, 1068, 1462, 1464]


In [3]:
def load_metrics(dataset_ids, task_type, metric_name, normalize=False, silent=False):
    """Load metrics, optionally print table, return arrays and DataFrame."""
    ebm_all, nam_all, gam_all, rows = [], [], [], []
    for ds_id in dataset_ids:
        fps = {m: results_dir / f"{m.lower()}_OpenML_{ds_id}_{task_type}_performance.json" for m in ['EBM', 'NAM', 'GAM']}
        metrics = {m: np.array([f['metric_value'] for f in json.load(open(fp))['folds']]) if fp.exists() else None for m, fp in fps.items()}
        if normalize and all(metrics[m] is not None for m in ['EBM', 'NAM', 'GAM']):
            c = np.concatenate(list(metrics.values()))
            m, s = c.mean(), c.std()
            for k in metrics: metrics[k] = (metrics[k] - m) / s if s > 0 else metrics[k] - m
            ebm_all.extend(metrics['EBM']); nam_all.extend(metrics['NAM']); gam_all.extend(metrics['GAM'])
        row = {'Dataset ID': ds_id}
        for m in ['EBM', 'NAM', 'GAM']:
            row[f'{m} Mean'] = metrics[m].mean() if metrics[m] is not None else np.nan
            row[f'{m} Std'] = metrics[m].std() if metrics[m] is not None else np.nan
        rows.append(row)
    df = pd.DataFrame(rows)
    if not silent:
        print("=" * 100, f"\n{task_type.upper()}: Performance Summary ({metric_name} ({'lower' if task_type == 'regression' else 'higher'} is better))", "=" * 100, "\nPer-Dataset Results:", pd.DataFrame({'Dataset ID': df['Dataset ID'], **{f'{m} ({metric_name})': df.apply(lambda r: f"{r[f'{m} Mean']:.4f} ± {r[f'{m} Std']:.4f}" if not np.isnan(r[f'{m} Mean']) else 'N/A', axis=1) for m in ['EBM', 'NAM', 'GAM']}}).to_string(index=False), "\nOverall Summary:", *[f"  {m}: {df[f'{m} Mean'].mean():.4f} ± {df[f'{m} Mean'].std():.4f}" if not np.isnan(df[f'{m} Mean'].mean()) else f"  {m}: N/A" for m in ['EBM', 'NAM', 'GAM']], "=" * 100, sep='\n')
    return np.array(ebm_all), np.array(nam_all), np.array(gam_all), df


In [ ]:
# Load and print performance tables
ebm_reg, nam_reg, gam_reg, regression_table = load_metrics(regression_datasets, 'regression', 'RMSE')
print()
ebm_clf, nam_clf, gam_clf, classification_table = load_metrics(classification_datasets, 'classification', 'AUC')


NameError: name 'load_per_dataset_metrics' is not defined

## Initial Friedman Test

Quick check if models differ significantly before detailed comparison.


In [ ]:
def run_friedman_test(ebm_data, nam_data, gam_data, task_name):
    """Run Friedman test and print results."""
    if len(ebm_data) > 0 and len(nam_data) > 0 and len(gam_data) > 0:
        friedman_stat, friedman_pval = friedmanchisquare(ebm_data, nam_data, gam_data)
        print(f"\n{task_name}:")
        print(f"  Statistic: {friedman_stat:.4f}")
        print(f"  p-value: {friedman_pval:.6f}")
        print(f"  Significant (α=0.05): {'Yes' if friedman_pval < 0.05 else 'No'}")
        if friedman_pval < 0.05:
            print(f"  → Models differ significantly. Proceed with detailed comparison.")
        else:
            print(f"  → No significant difference between models.")
    else:
        print(f"\n{task_name}: Cannot perform test (insufficient data)")

# Load normalized metrics for Friedman test (silent)
ebm_reg, nam_reg, gam_reg, _ = load_metrics(regression_datasets, 'regression', 'RMSE', normalize=True, silent=True)
ebm_clf, nam_clf, gam_clf, _ = load_metrics(classification_datasets, 'classification', 'AUC', normalize=True, silent=True)

print("=" * 70)
print("FRIEDMAN TEST: Overall Significance Check")
print("=" * 70)

run_friedman_test(ebm_reg, nam_reg, gam_reg, 'REGRESSION')
run_friedman_test(ebm_clf, nam_clf, gam_clf, 'CLASSIFICATION')

print("=" * 70)


FRIEDMAN TEST: Overall Significance Check

REGRESSION:
  Statistic: 7.5600
  p-value: 0.022823
  Significant (α=0.05): Yes
  → Models differ significantly. Proceed with detailed comparison.

CLASSIFICATION:
  Statistic: 10.2462
  p-value: 0.005958
  Significant (α=0.05): Yes
  → Models differ significantly. Proceed with detailed comparison.


## Detailed Three-Way Comparison

If Friedman test is significant, detailed comparison with Nemenyi post-hoc test follows.


In [ ]:
def run_statistical_test(ebm_data, nam_data, gam_data, task_type, table):
    """Run detailed comparison with Nemenyi post-hoc test, identify best model."""
    print("=" * 70)
    print(f"{task_type.upper()}: Detailed Three-Way Comparison (EBM vs NAM vs GAM)")
    print("=" * 70)
    
    if len(ebm_data) == 0 or len(nam_data) == 0 or len(gam_data) == 0:
        print("Cannot perform tests: insufficient data")
        return
    
    print(f"EBM mean (normalized): {ebm_data.mean():.4f} ± {ebm_data.std():.4f}")
    print(f"NAM mean (normalized): {nam_data.mean():.4f} ± {nam_data.std():.4f}")
    print(f"GAM mean (normalized): {gam_data.mean():.4f} ± {gam_data.std():.4f}")
    
    friedman_stat, friedman_pval = friedmanchisquare(ebm_data, nam_data, gam_data)
    print(f"\nFriedman test: statistic={friedman_stat:.4f}, p-value={friedman_pval:.6f}, "
          f"significant={'Yes' if friedman_pval < 0.05 else 'No'}")
    
    if friedman_pval < 0.05:
        from scipy.stats import rankdata
        
        # Prepare data for scikit-posthocs: DataFrame with columns as groups
        data_df = pd.DataFrame({
            'EBM': ebm_data,
            'NAM': nam_data,
            'GAM': gam_data
        })
        
        # Run Nemenyi post-hoc test
        nemenyi_pvals = sp.posthoc_nemenyi_friedman(data_df)
        
        # Calculate mean ranks
        ranks = np.zeros((len(ebm_data), 3))
        for i in range(len(ebm_data)):
            values = np.array([ebm_data[i], nam_data[i], gam_data[i]])
            ranks[i] = rankdata(values, method='average')
        mean_ranks_arr = ranks.mean(axis=0)
        mean_ranks = {'EBM': mean_ranks_arr[0], 'NAM': mean_ranks_arr[1], 'GAM': mean_ranks_arr[2]}
        
        print(f"\nNemenyi post-hoc test (p-values):")
        print(f"  Mean ranks: EBM={mean_ranks['EBM']:.4f}, NAM={mean_ranks['NAM']:.4f}, GAM={mean_ranks['GAM']:.4f}")
        print(f"  Pairwise p-values:")
        comparisons = [('EBM', 'NAM'), ('EBM', 'GAM'), ('NAM', 'GAM')]
        for m1, m2 in comparisons:
            pval = nemenyi_pvals.loc[m1, m2]
            sig_marker = "***" if pval < 0.05 else ""
            print(f"    {m1} vs {m2}: p={pval:.6f} {sig_marker}")
        
        best_model = (min if task_type == 'regression' else max)(mean_ranks, key=mean_ranks.get)
        print(f"\n  Best model: {best_model} ({'lowest' if task_type == 'regression' else 'highest'} mean rank = {mean_ranks[best_model]:.4f})")
    else:
        means = {m: table[f'{m} Mean'].mean() for m in ['EBM', 'NAM', 'GAM']}
        best_model = (min if task_type == 'regression' else max)(means, key=means.get)
        print(f"\n  Best model (by mean): {best_model}")
    print("=" * 70)

# Run detailed comparison (data already loaded in previous cell)
run_statistical_test(ebm_reg, nam_reg, gam_reg, 'regression', regression_table)
print()
run_statistical_test(ebm_clf, nam_clf, gam_clf, 'classification', classification_table)


REGRESSION: Detailed Three-Way Comparison (EBM vs NAM vs GAM)
EBM mean (normalized): -0.4691 ± 1.0953
NAM mean (normalized): 0.3219 ± 0.9163
GAM mean (normalized): 0.1472 ± 0.7844

Friedman test: statistic=7.5600, p-value=0.022823, significant=Yes

Nemenyi post-hoc test (p-values):
  Mean ranks: EBM=1.7000, NAM=2.2400, GAM=2.0600
  Pairwise p-values:
    EBM vs NAM: p=0.019004 ***
    EBM vs GAM: p=0.169555 
    NAM vs GAM: p=0.640299 

  Best model: EBM (lowest mean rank = 1.7000)

CLASSIFICATION: Detailed Three-Way Comparison (EBM vs NAM vs GAM)
EBM mean (normalized): 0.3602 ± 0.8126
NAM mean (normalized): -0.3504 ± 1.0713
GAM mean (normalized): -0.0098 ± 0.9692

Friedman test: statistic=10.2462, p-value=0.005958, significant=Yes

Nemenyi post-hoc test (p-values):
  Mean ranks: EBM=2.3300, NAM=1.7000, GAM=1.9700
  Pairwise p-values:
    EBM vs NAM: p=0.004652 ***
    EBM vs GAM: p=0.169555 
    NAM vs GAM: p=0.367506 

  Best model: EBM (highest mean rank = 2.3300)
